# HyperQwen-3.6-35B-A3B — Portfolio Validation Notebook

**Thesis**: HyperQwen-3.6-35B-A3B builds on FRIT (Anut et al., 2025), RAGLens (Xiong et al., ICLR 2026), and the Neural Probe / INSIDE literature, combining validated techniques with **novel mechanistic modules** for agent loop detection and saturation-based early stopping. We contribute first application to the Qwen3.6 MoE hybrid architecture, an activation-level extension of FRIT's causal intervention, and a production-ready toolkit integrating these capabilities.

**This notebook validates the NOVEL modules** (qwen-monitor + qwen-efficient) on Qwen3.6-35B-A3B in a single Colab RTX 6000 Blackwell session. Follow-up notebooks will port FRIT and RAGLens.

## Scope

| Section | What | Source | Expected time |
|---|---|---|---|
| **A — Foundation** | Load Qwen3.6-35B-A3B, regen probe L11 from Stage B corpus | Our prior work | 25 min |
| **B — qwen-monitor** | Synthetic agent trace detection via activation similarity | **Novel** | 1.5 h |
| **C — qwen-efficient** | Saturation-based early-stop on Stage B corpus | **Novel** | 30 min |
| **D — Report** | Results table + HF upload of artifacts | — | 5 min |

**Total: ~2.5-3 h on RTX 6000 Blackwell 96 GB.**

## Prior art we build on (must be cited in any publication)

- **FRIT** — Anut-py, *Using Causal Importance to Improve Chain-of-Thought Faithfulness* (arxiv:2509.13334) — text-level intervention + DPO training, our CCR extends to activation-level
- **RAGLens** — Xiong et al., ICLR 2026 (arxiv:2512.08892) — SAE + GAM for RAG grounding, our toolkit ports to Qwen3.6 MoE
- **Neural Probe-Based Hallucination Detection** (arxiv:2512.20949) — MLP probes on Qwen2.5-7B
- **INSIDE** — Chen et al., ICLR 2024 — internal state for hallucination
- **Obfuscation Atlas** — FAR AI, Feb 2026 (arxiv:2602.15515) — frozen probe as RL penalty for deception
- **Stage B corpus** — our own: `caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b` (1000 labeled rollouts with L11/L17/L23 activations per response token)

## Cell 1 — Install

Same stack validated in prior Colab sessions: transformers main (Qwen3.5-MoE support), fla (GDN fast path), causal-conv1d (prebuilt wheel for torch 2.10 ABI compat).

In [ ]:
import sys, subprocess, os, shutil, site
from pathlib import Path

def _pip(*a, check=True):
    return subprocess.run([sys.executable, '-m', 'pip', *a], check=check)

def _check():
    try:
        import transformers
        try:
            from transformers.models.auto.configuration_auto import CONFIG_MAPPING_NAMES
        except ImportError:
            from transformers.models.auto.auto_mappings import CONFIG_MAPPING_NAMES
        return 'qwen3_5_moe' in CONFIG_MAPPING_NAMES, transformers.__version__
    except Exception as e:
        return False, str(e)

ok, ver = _check()
print(f'Current transformers: {ver}, qwen3_5_moe={ok}')

if not ok:
    _pip('install', '-q', 'accelerate', 'datasets', 'huggingface_hub==1.5.0',
         'safetensors', 'einops', 'scikit-learn', 'sentencepiece', 'tokenizers',
         'protobuf', 'scipy', 'matplotlib')
    _pip('uninstall', '-y', 'transformers', check=False)
    SRC = '/content/transformers_src'
    if Path(SRC).exists(): shutil.rmtree(SRC)
    subprocess.run(['git','clone','--quiet','https://github.com/huggingface/transformers.git',SRC], check=True)
    _pip('install','--force-reinstall','--no-deps','--no-cache-dir', SRC)
    _pip('install','--no-cache-dir','flash-linear-attention', check=False)
    _pip('install','-q','https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl', check=False)
    for m in list(sys.modules):
        if m.startswith('transformers') or m.startswith('huggingface_hub'): del sys.modules[m]
    ok, ver = _check()
    print(f'Post-install: {ver}, qwen3_5_moe={ok}')
    if not ok:
        print('⚠️ Restart kernel + rerun cell')
        raise SystemExit

try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
    if hf_token:
        from huggingface_hub import login
        login(token=hf_token)
        print('✅ HF authenticated')
except Exception as e:
    print(f'(HF auth skipped: {e})')

import json, time, re, gc, pickle, random
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt

OUT = Path('/content/hyperqwen')
OUT.mkdir(exist_ok=True)
print(f'workspace: {OUT}')

## Cell 2 — CFG

In [ ]:
CFG = dict(
    model_id='Qwen/Qwen3.6-35B-A3B',
    stage_b_repo='caiovicentino1/Qwen3.6-35B-A3B-mcr-stage-b',
    capture_layers=[11, 17, 23],
    # qwen-monitor
    n_agent_traces=10,
    n_steps_per_trace=5,
    max_new_per_step=200,
    stuck_threshold=0.95,
    # qwen-efficient
    n_rollouts_saturation=100,
    sat_cos_threshold=0.80,
    sat_window=32,
)
print(json.dumps(CFG, indent=2))

# SECTION A — Foundation

Load Qwen3.6-35B-A3B and regenerate our probe artifacts from the Stage B corpus. This validates our infrastructure works on the model.

## Cell 3 — Load Qwen3.6-35B-A3B

In [ ]:
from transformers import AutoTokenizer, AutoModelForImageTextToText

tok = AutoTokenizer.from_pretrained(CFG['model_id'], trust_remote_code=True)
if tok.pad_token_id is None: tok.pad_token = tok.eos_token

if 'model' not in dir():
    model = AutoModelForImageTextToText.from_pretrained(
        CFG['model_id'], dtype=torch.bfloat16, device_map='cuda',
        attn_implementation='sdpa', trust_remote_code=True)
    model.eval()
print(f'VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB / {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB')

## Cell 4 — Regen probe L11 (our foundation)

Pull Stage B corpus, fit probe on 80% / validate on 20%. Reproduces prior validated AUROC ~0.78 deterministically (seed=42).

In [ ]:
from huggingface_hub import snapshot_download
from safetensors import safe_open
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

corpus = snapshot_download(CFG['stage_b_repo'], repo_type='dataset', cache_dir='/content/cache')
shard_dir = Path(corpus) / 'shards'
shards = sorted(shard_dir.glob('q*.safetensors'))
print(f'{len(shards)} shards')

pooled = {li: [] for li in CFG['capture_layers']}
outcomes = []
for shard in tqdm(shards, desc='pool'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])
        rollouts = json.loads(meta['rollouts'])
        for li in CFG['capture_layers']:
            acts = f.get_tensor(f'acts_L{li}')
            o = offs[f'L{li}']
            for r in range(len(rollouts)):
                pooled[li].append(acts[o[r]:o[r+1]].float().mean(0).numpy())
        outcomes.extend(int(r['correct']) for r in rollouts)

outcomes = np.array(outcomes[:1000])
for li in CFG['capture_layers']:
    pooled[li] = np.stack(pooled[li])
print(f'pooled shapes: {pooled[11].shape}, correct: {outcomes.sum()}/{len(outcomes)}')

idx_fit, idx_val = train_test_split(np.arange(1000), test_size=0.2, stratify=outcomes, random_state=42)
models_d = {}
for li in CFG['capture_layers']:
    scaler = StandardScaler().fit(pooled[li][idx_fit])
    Xf = scaler.transform(pooled[li][idx_fit])
    Xv = scaler.transform(pooled[li][idx_val])
    pca = PCA(n_components=64, random_state=42).fit(Xf)
    lr = LogisticRegression(C=1.0, max_iter=2000, random_state=42).fit(pca.transform(Xf), outcomes[idx_fit])
    auroc = roc_auc_score(outcomes[idx_val], lr.decision_function(pca.transform(Xv)))
    print(f'  L{li}: AUROC={auroc:.4f}')
    models_d[f'L{li}'] = dict(scaler=scaler, pca=pca, logreg=lr)

# Isotonic calibrator on L11
l11 = models_d['L11']
probe_raw = l11['logreg'].predict_proba(l11['pca'].transform(l11['scaler'].transform(pooled[11][idx_fit])))[:,1]
iso = IsotonicRegression(out_of_bounds='clip').fit(probe_raw, outcomes[idx_fit])
print('✅ probe + calibrator fitted')

with open(OUT / 'probe_bundle.pkl', 'wb') as f:
    pickle.dump(dict(models_d=models_d, calibrator=iso), f)
print(f'✅ saved {OUT}/probe_bundle.pkl')

## Cell 5 — Hook utilities (shared by all modules)

In [ ]:
def get_layer_module(m, idx):
    cands = [m]
    if hasattr(m, 'model'): cands.append(m.model)
    for start in cands:
        for path in [('model','language_model','layers'),('language_model','layers'),('model','layers'),('layers',)]:
            cur = start; ok = True
            for p in path:
                if hasattr(cur, p): cur = getattr(cur, p)
                else: ok = False; break
            if ok and hasattr(cur, '__getitem__'):
                try: return cur[idx]
                except: continue
    raise RuntimeError(f'layer {idx} not found')

class MultiCap:
    def __init__(self, layers): self.h = {i: None for i in layers}
    def make_hook(self, idx):
        def h(m, i, o):
            self.h[idx] = (o[0] if isinstance(o,tuple) else o).detach()
        return h

def register_hooks(m, layers):
    cap = MultiCap(layers)
    handles = [get_layer_module(m,i).register_forward_hook(cap.make_hook(i)) for i in layers]
    return cap, handles

def pool_response(acts, prompt_len, response_len):
    if response_len == 0:
        return torch.zeros(acts.shape[-1], device=acts.device)
    return acts[0, prompt_len:prompt_len+response_len].float().mean(0)

# Sanity: probe the hooks
cap, handles = register_hooks(model, CFG['capture_layers'])
with torch.inference_mode():
    probe_in = torch.randint(0, 1000, (1, 16), device='cuda')
    _ = model(input_ids=probe_in, attention_mask=torch.ones_like(probe_in), use_cache=False)
for li in CFG['capture_layers']:
    assert cap.h[li] is not None, f'hook L{li} silent'
    print(f'  L{li}: {tuple(cap.h[li].shape)}')
d_model = cap.h[CFG['capture_layers'][0]].shape[-1]
for h in handles: h.remove()
print(f'✅ hooks OK, d_model={d_model}')

# SECTION B — qwen-monitor (NOVEL)

**Hypothesis**: When an agent is stuck in a loop, even if the output text differs, the internal L23 residual state is highly similar across steps. We can detect "stuck" mechanistically **before** output-level observability tools (MAST, Langfuse, MLflow) notice.

**Why this is novel**: no existing agent observability tool uses activation-level signal. Circuit breakers rely on output heuristics (consecutive failures, token velocity). Our claim: activation similarity separates stuck from progressing traces, testable empirically.

**Protocol**:
- Generate 10 synthetic agent traces: 5 stuck (same/equivalent prompts), 5 progressing (distinct subtasks)
- For each step: capture pooled L23 residual, compute cos_sim with prior 4 steps
- Classify: stuck if max(recent_sims) > threshold
- Report: AUROC of stuck_score vs ground truth

## Cell 6 — Agent trace templates + execution

In [ ]:
STUCK_TEMPLATES = [
    # Each element = list of N identical/equivalent prompts (stuck in loop)
    ["Agent step {i}: What is 17 + 25?"] * 5,
    ["Agent step {i}: Who wrote Hamlet?"] * 5,
    ["Agent step {i}: Convert 20 celsius to fahrenheit."] * 5,
    [
        "Agent step {i}: Calculate sqrt(144).",
        "Agent step {i}: What is the square root of 144?",
        "Agent step {i}: Compute the square root of one hundred forty four.",
        "Agent step {i}: Square root of 144 equals?",
        "Agent step {i}: sqrt(144) = ?",
    ],
    [
        "Agent step {i}: List three primary colors.",
        "Agent step {i}: What are three primary colors?",
        "Agent step {i}: Name three colors that are primary.",
        "Agent step {i}: Three colors considered primary?",
        "Agent step {i}: Enumerate the primary colors (three).",
    ],
]

PROGRESSING_TEMPLATES = [
    # Each element = list of N genuinely different subtasks (progressing)
    [
        "Agent step {i}: Calculate 17 + 25.",
        "Agent step {i}: Multiply the previous result by 3.",
        "Agent step {i}: Subtract 12 from that.",
        "Agent step {i}: Divide the result by 2.",
        "Agent step {i}: State the final answer rounded to integer.",
    ],
    [
        "Agent step {i}: List three primary colors.",
        "Agent step {i}: Pick one color and name a famous painting using it.",
        "Agent step {i}: Who painted it?",
        "Agent step {i}: In which year?",
        "Agent step {i}: What museum houses it today?",
    ],
    [
        "Agent step {i}: Define entropy in information theory.",
        "Agent step {i}: Give the formula for Shannon entropy.",
        "Agent step {i}: Compute entropy of a fair coin flip in bits.",
        "Agent step {i}: What does higher entropy imply?",
        "Agent step {i}: Name a real-world use case of entropy.",
    ],
    [
        "Agent step {i}: Who wrote Hamlet?",
        "Agent step {i}: When was Hamlet written (approximate year)?",
        "Agent step {i}: Name one major character other than Hamlet.",
        "Agent step {i}: What city is the play set in?",
        "Agent step {i}: In one sentence summarize the plot.",
    ],
    [
        "Agent step {i}: Convert 20 celsius to fahrenheit.",
        "Agent step {i}: Now convert 100 celsius to fahrenheit.",
        "Agent step {i}: What is the difference between those two fahrenheit values?",
        "Agent step {i}: Express that difference in kelvins.",
        "Agent step {i}: Is a 80F change considered a large daily swing? One-word answer.",
    ],
]

def run_trace(prompts, trace_type, trace_id):
    """Run N-step agent trace, return L23 pooled activations per step + texts."""
    assert len(prompts) == CFG['n_steps_per_trace']
    cap, handles = register_hooks(model, [23])
    pools = []
    texts = []
    try:
        for step_i, prompt_tpl in enumerate(prompts):
            prompt = prompt_tpl.format(i=step_i+1)
            enc = tok(prompt, return_tensors='pt').to('cuda')
            P = enc.input_ids.shape[1]
            cap.h[23] = None
            with torch.no_grad():
                gen = model.generate(**enc, max_new_tokens=CFG['max_new_per_step'],
                                      do_sample=False, pad_token_id=tok.pad_token_id,
                                      use_cache=True)
            # Second forward with hook to capture activations
            attn = (gen != tok.pad_token_id).long()
            with torch.no_grad():
                model(input_ids=gen, attention_mask=attn, use_cache=False)
            response_len = gen.shape[1] - P
            pool_L23 = pool_response(cap.h[23], P, response_len).cpu().numpy()
            pools.append(pool_L23)
            texts.append(tok.decode(gen[0,P:], skip_special_tokens=True))
    finally:
        for h in handles: h.remove()
    return dict(type=trace_type, id=trace_id, pools=pools, texts=texts, prompts=[p.format(i=i+1) for i,p in enumerate(prompts)])

# Execute all traces
traces = []
t0 = time.time()
for i, prompts in enumerate(STUCK_TEMPLATES):
    print(f'stuck trace {i+1}/5...')
    traces.append(run_trace(prompts, 'stuck', i))
for i, prompts in enumerate(PROGRESSING_TEMPLATES):
    print(f'progressing trace {i+1}/5...')
    traces.append(run_trace(prompts, 'progressing', i))
print(f'\n✅ all 10 traces done in {(time.time()-t0)/60:.1f} min')

## Cell 7 — Compute step-to-step similarity

In [ ]:
def cos(a, b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b) + 1e-8))

for t in traces:
    pools = t['pools']
    t['sims_pairwise'] = [cos(pools[i], pools[i-1]) for i in range(1, len(pools))]
    t['mean_sim'] = float(np.mean(t['sims_pairwise']))
    t['max_sim'] = float(np.max(t['sims_pairwise']))

# Summarize
stuck_means = [t['mean_sim'] for t in traces if t['type']=='stuck']
prog_means = [t['mean_sim'] for t in traces if t['type']=='progressing']
print('STUCK traces:')
for t in traces:
    if t['type']=='stuck':
        print(f'  id{t["id"]}: mean_sim={t["mean_sim"]:.4f}  max={t["max_sim"]:.4f}  [sims: {[round(s,3) for s in t["sims_pairwise"]]}]')
print(f'  STUCK mean: {np.mean(stuck_means):.4f}')
print('\nPROGRESSING traces:')
for t in traces:
    if t['type']=='progressing':
        print(f'  id{t["id"]}: mean_sim={t["mean_sim"]:.4f}  max={t["max_sim"]:.4f}  [sims: {[round(s,3) for s in t["sims_pairwise"]]}]')
print(f'  PROG  mean: {np.mean(prog_means):.4f}')
gap = np.mean(stuck_means) - np.mean(prog_means)
print(f'\nΔ(stuck - prog) = {gap:+.4f}  (positive = detector works)')

## Cell 8 — Classifier performance + plot

Use mean_sim as stuck score. AUROC over all 10 traces.

**Decision rules**:
- AUROC > 0.85 → qwen-monitor validated, ship in toolkit
- AUROC 0.70-0.85 → useful, needs threshold tuning
- AUROC < 0.70 → signal too weak, revisit formulation

In [ ]:
y_true = np.array([1 if t['type']=='stuck' else 0 for t in traces])
scores = np.array([t['mean_sim'] for t in traces])
auroc_monitor = roc_auc_score(y_true, scores)

# Best threshold (Youden J)
thresholds = np.linspace(scores.min(), scores.max(), 50)
best_th, best_j = 0, 0
for th in thresholds:
    preds = (scores > th).astype(int)
    tp = ((preds==1) & (y_true==1)).sum()
    tn = ((preds==0) & (y_true==0)).sum()
    fp = ((preds==1) & (y_true==0)).sum()
    fn = ((preds==0) & (y_true==1)).sum()
    if tp+fn==0 or tn+fp==0: continue
    j = tp/(tp+fn) - fp/(fp+tn)
    if j > best_j:
        best_j = j; best_th = th

preds = (scores > best_th).astype(int)
accuracy = (preds == y_true).mean()

print(f'qwen-monitor AUROC: {auroc_monitor:.4f}')
print(f'Best threshold: {best_th:.4f}  (Youden J={best_j:.3f})')
print(f'Accuracy at best threshold: {100*accuracy:.1f}%')
verdict = 'SHIP' if auroc_monitor > 0.85 else ('TUNE' if auroc_monitor > 0.70 else 'REVISIT')
print(f'Verdict: {verdict}')

# Plot
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist([t['mean_sim'] for t in traces if t['type']=='stuck'], bins=10, alpha=0.6, label='stuck', color='#e74c3c')
ax[0].hist([t['mean_sim'] for t in traces if t['type']=='progressing'], bins=10, alpha=0.6, label='progressing', color='#2ecc71')
ax[0].axvline(best_th, color='k', ls='--', label=f'threshold={best_th:.3f}')
ax[0].set_xlabel('mean cos_sim step-to-step (L23 pooled)')
ax[0].set_ylabel('count')
ax[0].set_title(f'qwen-monitor: AUROC={auroc_monitor:.3f}')
ax[0].legend()

for t in traces:
    color = '#e74c3c' if t['type']=='stuck' else '#2ecc71'
    ax[1].plot(range(1, len(t['sims_pairwise'])+1), t['sims_pairwise'], 'o-', color=color, alpha=0.6,
               label=t['type'] if t['id']==0 else None)
ax[1].set_xlabel('step pair (N vs N-1)')
ax[1].set_ylabel('cos_sim')
ax[1].set_title('per-trace step-to-step similarity')
ax[1].legend()
plt.tight_layout()
plt.savefig(OUT/'qwen_monitor_results.png', dpi=120, bbox_inches='tight')
plt.show()

with open(OUT/'results_monitor.json','w') as f:
    json.dump(dict(
        auroc=float(auroc_monitor), best_threshold=float(best_th), accuracy=float(accuracy),
        verdict=verdict,
        stuck_means=[float(s) for s in stuck_means], prog_means=[float(s) for s in prog_means],
        traces=[dict(type=t['type'], id=t['id'], mean_sim=t['mean_sim'], sims=t['sims_pairwise']) for t in traces]
    ), f, indent=2)
print(f'✅ saved {OUT}/results_monitor.json + qwen_monitor_results.png')

# SECTION C — qwen-efficient (NOVEL)

**Hypothesis**: L23 residual trajectory reaches stability (saturation) before `\boxed{}` commit in correct responses, enabling early stopping.

**Why this is novel**: no prior work uses SAE feature convergence / activation stability as an RL or inference-time early-stopping signal.

**Protocol** (reuses Stage B corpus — pure CPU analysis):
- For 100 rollouts (mix correct/wrong): compute L23 rolling-window stability
- Identify saturation point as first t where `cos_sim(window_t, window_t+w) ≥ 0.80` for 32 consecutive t
- Compare saturation point in correct vs wrong rollouts
- Decision rule: if `saturation_correct` < `saturation_wrong` by ≥20%, ship early-stop feature

**Known prior finding**: our Stage B analysis showed saturation formulation was weak (no plateau at 0.80 threshold). This cell re-verifies in clean env + explores alternative formulations (response length inverse correlation ρ=-0.31 was actually the real signal).

In [ ]:
def stability_point(acts, cos_thresh=CFG['sat_cos_threshold'], W=CFG['sat_window'], min_start=100):
    """First t where adjacent 32-tok window means keep cos_sim > threshold for W consecutive t."""
    L, d = acts.shape
    if L < min_start + 2*W + W: return L
    means = np.stack([acts[t:t+W].mean(0) for t in range(L - W + 1)])
    means_n = means / (np.linalg.norm(means, axis=-1, keepdims=True) + 1e-8)
    sim = (means_n[:-W] * means_n[W:]).sum(axis=-1)
    above = sim >= cos_thresh
    for t in range(min_start, len(above) - W):
        if above[t:t+W].all():
            return t + W
    return L

# Sample 100 rollouts from Stage B corpus (first N shards, mix of correct/wrong)
sample_shards = shards[:20]  # 20 questions × 5 rollouts = 100 rollouts
rollouts_data = []
for shard in tqdm(sample_shards, desc='analyze'):
    with safe_open(str(shard), framework='pt') as f:
        meta = f.metadata()
        offs = json.loads(meta['offsets'])['L23']
        rollouts = json.loads(meta['rollouts'])
        acts_L23 = f.get_tensor('acts_L23').float().numpy()
        for r_idx in range(len(rollouts)):
            acts_r = acts_L23[offs[r_idx]:offs[r_idx+1]]
            L = acts_r.shape[0]
            if L < 100: continue
            sp = stability_point(acts_r)
            rollouts_data.append(dict(
                shard=shard.name, rollout=r_idx,
                response_len=L, saturation_point=int(sp),
                saturated=bool(sp < L),
                wasted_tokens=max(0, L - sp),
                correct=bool(rollouts[r_idx]['correct']),
            ))

df_ef = rollouts_data
print(f'\nanalyzed {len(df_ef)} rollouts')
print(f'saturation triggered (before end): {sum(r["saturated"] for r in df_ef)}/{len(df_ef)}')
correct_wasted = [r['wasted_tokens'] for r in df_ef if r['correct']]
wrong_wasted = [r['wasted_tokens'] for r in df_ef if not r['correct']]
print(f'median wasted (correct): {np.median(correct_wasted) if correct_wasted else 0:.0f}')
print(f'median wasted (wrong):   {np.median(wrong_wasted) if wrong_wasted else 0:.0f}')

# Check also response_len inverse correlation (known from Stage B to be real signal)
from scipy.stats import spearmanr
lens = np.array([r['response_len'] for r in df_ef])
corr = np.array([int(r['correct']) for r in df_ef])
rho, p = spearmanr(lens, corr)
print(f'\nSpearman ρ(response_len, correct): {rho:+.4f}  p={p:.2e}  (known from Stage B ≈ -0.31)')

# Binary completion indicator (the ACTUAL working formulation from Stage B)
MAX_NEW = 2048  # Stage B used max_new=2048
committed = np.array([r['response_len'] < MAX_NEW for r in df_ef])
acc_committed = corr[committed].mean() if committed.sum() else 0
acc_maxed = corr[~committed].mean() if (~committed).sum() else 0
print(f'\nbinary commit indicator (from Stage B — KNOWN effective):')
print(f'  committed ({committed.sum()} rollouts): {100*acc_committed:.1f}% correct')
print(f'  maxed     ({(~committed).sum()} rollouts): {100*acc_maxed:.1f}% correct')
gap_pp = 100 * (acc_committed - acc_maxed)
print(f'  gap: {gap_pp:+.1f}pp (Stage B saw +42pp)')

verdict_efficient = 'SHIP' if gap_pp > 30 else ('TUNE' if gap_pp > 15 else 'REVISIT')
print(f'\nVerdict qwen-efficient: {verdict_efficient}')

with open(OUT/'results_efficient.json','w') as f:
    json.dump(dict(
        n_rollouts=len(df_ef),
        saturation_triggered_frac=float(sum(r['saturated'] for r in df_ef)/len(df_ef)),
        median_wasted_correct=float(np.median(correct_wasted)) if correct_wasted else None,
        median_wasted_wrong=float(np.median(wrong_wasted)) if wrong_wasted else None,
        response_len_correlation=float(rho),
        response_len_correlation_p=float(p),
        binary_commit_gap_pp=float(gap_pp),
        verdict=verdict_efficient,
    ), f, indent=2)
print(f'✅ saved {OUT}/results_efficient.json')

# SECTION D — Final report

Summarize results, mark go/no-go for both modules.

In [ ]:
report = dict(
    timestamp=time.strftime('%Y-%m-%d %H:%M:%S'),
    model='Qwen/Qwen3.6-35B-A3B',
    foundation=dict(
        probe_L11_AUROC=float(roc_auc_score(outcomes[idx_val],
                                            models_d['L11']['logreg'].decision_function(
                                                models_d['L11']['pca'].transform(
                                                    models_d['L11']['scaler'].transform(pooled[11][idx_val]))))),
    ),
    qwen_monitor=dict(
        auroc=float(auroc_monitor),
        best_threshold=float(best_th),
        accuracy=float(accuracy),
        verdict=verdict,
    ),
    qwen_efficient=dict(
        binary_commit_gap_pp=float(gap_pp),
        response_len_correlation=float(rho),
        verdict=verdict_efficient,
    ),
    cites=dict(
        FRIT='arxiv:2509.13334 (Anut et al., 2025)',
        RAGLens='arxiv:2512.08892 / ICLR 2026 (Xiong et al.)',
        NeuralProbe='arxiv:2512.20949 (2025)',
        INSIDE='Chen et al., ICLR 2024',
    ),
)
print(json.dumps(report, indent=2))

with open(OUT/'hyperqwen_portfolio_report.json','w') as f:
    json.dump(report, f, indent=2)
print(f'\n✅ saved {OUT}/hyperqwen_portfolio_report.json')

print('\n' + '='*60)
print('  HYPERQWEN PORTFOLIO — NEXT STEPS')
print('='*60)
print(f'  qwen-monitor: {verdict} (AUROC={auroc_monitor:.3f})')
print(f'  qwen-efficient: {verdict_efficient} (gap={gap_pp:+.1f}pp)')
print()
print('If both SHIP: proceed to port FRIT + RAGLens in follow-up notebooks')
print('If TUNE/REVISIT on one: diagnose before expanding toolkit')

## (Opcional) Upload dos artefatos pro HF

Se quiser preservar os resultados.

In [ ]:
# UPLOAD = False  # set True pra subir pro HF
UPLOAD = False
if UPLOAD:
    from huggingface_hub import HfApi, create_repo
    repo_id = 'caiovicentino1/hyperqwen-3.6-35b-a3b-validation'
    create_repo(repo_id, repo_type='dataset', private=True, exist_ok=True)
    api = HfApi()
    api.upload_folder(folder_path=str(OUT), repo_id=repo_id, repo_type='dataset',
                      commit_message='portfolio validation results')
    print(f'✅ uploaded to hf.co/datasets/{repo_id}')
else:
    print('skipped (set UPLOAD=True to push to HF)')